# 09.4 · Exercises

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/09-nlp-genai/04-attention-transformers/exercises.ipynb)

Exercises 1 and 3 on paper first.

In [ ]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import os, logging, warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, str(data_path("download.py").parent))

In [ ]:
words = ["the", "mug", "was", "broken"]
Q = np.array([[0, 1], [1, 0], [0, 1], [1, 1]], dtype=float)
K = np.array([[1, 0], [0, 2], [1, 0], [0, 1]], dtype=float)
V = np.array([[1, 0], [0, 4], [1, 0], [2, 2]], dtype=float)

def softmax(scores):
    e = np.exp(scores - scores.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

## Exercise 1: attention for *mug*

By hand: scores, softmax weights and output for *mug*. Then check with NumPy.

In [ ]:
# your code here

## Exercise 2: forget to divide

Multiply all of *broken*'s scores by 10 before the softmax. What happens to the weights?

In [ ]:
# your code here

## Exercise 3: causal mask

By hand: apply the causal mask to *mug*'s row and recompute its weights and output.

In [ ]:
# your code here

## Exercise 4: what does *it* refer to?

In DistilBERT, look at the attention from *it* in *"the mug fell off the shelf because it was fragile"*. Which layer and head put the most weight on *mug*? On *shelf*?

In [ ]:
import transformers
transformers.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()

from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")
bert = AutoModel.from_pretrained("distilbert/distilbert-base-uncased",
                                 output_attentions=True, attn_implementation="eager")

def attention(sentence):
    """Tokens and attention weights: array of shape (layers, heads, tokens, tokens)."""
    enc = tokenizer(sentence, return_tensors="pt")
    att = np.stack([a[0].detach().numpy() for a in bert(**enc).attentions])
    return tokenizer.convert_ids_to_tokens(enc["input_ids"][0]), att

In [ ]:
# your code here